# 00 · Diseño y gobierno antes de ingerir

**Objetivo:** relacionar las tablas de WideWorldImporters con el modelo dimensional y acordar responsabilidades.

**Modalidad:** material para explicar en clase. Los ejemplos se ejecutan únicamente en memoria, con datos ficticios. No conectan a una base, no crean archivos ni ejecutan una ingesta. Los bloques SQL son ilustrativos y no se envían a ningún servidor.

**Prerrequisitos:** Python con pandas. Cada notebook es independiente: no requiere scripts `.py` ni haber ejecutado otra notebook.


In [4]:
# Importar conexión centralizada desde el módulo
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, SERVIDOR, BD_ORIGEN, BD_DESTINO, USUARIO

# Obtener los engines
engine_origen, engine_destino = obtener_engines()
print(f'Conexiones establecidas correctamente.')
print(f'Servidor: {SERVIDOR}')
print(f'Origen: {BD_ORIGEN}')
print(f'Destino: {BD_DESTINO}')


Conexion OK (origen): WideWorldImporters
Conexion OK (destino): WideWorldImportersDW2
Conexiones establecidas correctamente.
Servidor: localhost\SQLEXPRESS02
Origen: WideWorldImporters
Destino: WideWorldImportersDW2


## Arquitectura que vamos a explicar

```text
WideWorldImporters -> RAW -> Calidad -> STAGING -> Dimensiones -> Hechos
                              |
                              v
                         Cuarentena -> Corrección -> Revalidación

Gobierno, catálogo, auditoría y linaje acompañan todo el recorrido.
```

Origen conceptual: `WideWorldImporters`. Destino conceptual:
`WideWorldImportersDW2`. No se crean ni se utilizan estas bases en la práctica.
El dominio es ventas, no toda la operación de la empresa.


In [5]:
import pandas as pd

mapa = pd.DataFrame([
    ("Sales.Customers", "dim_cliente", "Cliente y referencias"),
    ("Sales.CustomerCategories", "dim_cliente", "Categoría"),
    ("Sales.BuyingGroups", "dim_cliente", "Grupo de compra"),
    ("Warehouse.StockItems", "dim_producto", "Producto"),
    ("Purchasing.Suppliers", "dim_producto", "Proveedor"),
    ("Warehouse.Colors", "dim_producto", "Color"),
    ("Warehouse.PackageTypes", "dim_producto / control del detalle", "Empaque"),
    ("Application.People", "dim_vendedor", "Personas con rol vendedor"),
    ("Application.Cities", "dim_ciudad / cliente", "Ciudad"),
    ("Application.StateProvinces", "dim_ciudad", "Provincia"),
    ("Application.Countries", "dim_ciudad", "País"),
    ("Sales.Invoices", "fact_ventas", "Cabecera: cliente, fecha, vendedor"),
    ("Sales.InvoiceLines", "fact_ventas", "Detalle: producto y medidas"),
], columns=["origen", "destino", "aporte"])
display(mapa)
print("Tablas de origen:", len(mapa))


,origen,destino,aporte
0,Sales.Customers,dim_cliente,Cliente y referencias
1,Sales.CustomerCategories,dim_cliente,Categoría
2,Sales.BuyingGroups,dim_cliente,Grupo de compra
3,Warehouse.StockItems,dim_producto,Producto
4,Purchasing.Suppliers,dim_producto,Proveedor
5,Warehouse.Colors,dim_producto,Color
6,Warehouse.PackageTypes,dim_producto / control del detalle,Empaque
7,Application.People,dim_vendedor,Personas con rol vendedor
8,Application.Cities,dim_ciudad / cliente,Ciudad
9,Application.StateProvinces,dim_ciudad,Provincia


Tablas de origen: 13


## Grano, claves y responsables

Una fila de `fact_ventas` representa un `InvoiceLineID`. Una factura con tres
líneas genera tres hechos. `dim_tiempo` se genera desde un calendario.

- **NK:** identificador del sistema de origen.
- **SK:** identificador de una versión dimensional del warehouse.
- **Owner:** acuerda significado, uso y umbrales.
- **Steward:** revisa calidad y autoriza correcciones.
- **Engineer:** implementaría los controles en un sistema real.
- **CDO:** define políticas; **DBA:** aplica permisos técnicos.


In [6]:
catalogo = pd.DataFrame([
    ("Sales.Customers", "CustomerID", "Identificador del cliente", "Interno", "Owner Ventas"),
    ("Application.People", "FullName", "Nombre del vendedor", "PII / restringido", "Owner Ventas"),
    ("Sales.InvoiceLines", "ExtendedPrice", "Total con impuestos", "Confidencial", "Owner Ventas"),
], columns=["tabla", "columna", "significado", "clasificacion", "responsable"])
display(catalogo)


,tabla,columna,significado,clasificacion,responsable
0,Sales.Customers,CustomerID,Identificador del cliente,Interno,Owner Ventas
1,Application.People,FullName,Nombre del vendedor,PII / restringido,Owner Ventas
2,Sales.InvoiceLines,ExtendedPrice,Total con impuestos,Confidencial,Owner Ventas


## Gobierno no es solo documentación

Un catálogo no concede ni revoca permisos. En producción harían falta usuarios
separados para lectura de origen y escritura del destino, cifrado, retención y
auditoría. Aquí se explican esas decisiones, **no se implementa seguridad**.

Extraer únicamente las columnas necesarias reduce exposición. No convierte los
datos en anónimos: el nombre del vendedor sigue siendo PII.


## Actividad de discusión

¿Quién decide si se admite un 1% de rechazos? **Respuesta orientativa:** el Owner acuerda el impacto de negocio; el Steward formaliza y monitorea; el Engineer no debería inventar el umbral.